# 🧠 CerebroSpheroid V5: Cloud TPU High-Throughput Engine
### 105.9M V5 Small Architecture • 100% Loss-Bearing Contiguous Pretraining • Hardware Auto-Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_TPU_JAX.ipynb)

- **Pure JAX on Cloud TPU**: Native bfloat16 systolic MXU acceleration (~100,000–350,000 tok/s on TPU v5e / v6e).
- **Hardware Auto-Tuning**: Automatically discovers and tunes peak micro-batches and remat policy on active TPU hardware.
- **100% Loss-Bearing Contiguous Tokens**: Zero prompt-masking waste; unbroken 4,096-token sequence blocks.
- **Stage 1 Pretraining Mixture**: 85% FineWeb (sample-10BT) broad web + 15% Code (The Stack v2).
- **Remote Cloud Sync**: Real-time checkpoint mirroring (`latest.pt`, `best.pt`) and loss tracking on `bioflo.in` (Option A ~229 MB weights-only fast upload).

---
### ⚡ Colab Hardware Setting:
**Runtime** -> **Change runtime type** -> **TPU v5e** (TPU v5e-1) or **TPU v6e** (Trillium)

## ⚡ One-Click Cloud TPU Execution
Run the cell below with **Shift + Enter**. It automatically verifies the TPU v5 lite environment, 
pulls the latest hardened codebase, configures your cloud credentials, and launches contiguous pretraining.


In [ ]:
# ==============================================================================
# 🚀 ONE-CLICK RUN: CEREBROSPHEROID V5 CLOUD TPU HIGH-THROUGHPUT ENGINE
# ==============================================================================
# Setup transparent hugepages, install dependencies, pull latest code, authenticate,
# mount persistent storage, and launch JAX TPU pretraining with 1 click.
# ==============================================================================

# 1. System Dependencies & Hugepages
!sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled" 2>/dev/null || true
!pip install -q transformers datasets sentencepiece optax torch pyarrow huggingface_hub

import os, sys, subprocess, importlib, jax, urllib.request, json

# 2. Synchronize Repository to latest origin/main
REPO_DIR = "/content/cerebro_v5"
if not os.path.exists(REPO_DIR):
    print("📥 Cloning cerebro_spheroid_v5 repository...")
    subprocess.run(["git", "clone", "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git", REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Clear module caches
importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith(("src", "training", "autotune", "colab")):
        del sys.modules[mod]

# 3. Scratch Checkpoint Storage Setup (Zero Google Drive Dependency)
os.makedirs("/content/checkpoints", exist_ok=True)
print("ℹ️ Fast local scratch storage initialized (Checkpoints stream directly to bioflo.in).")


# 4. Dynamic Token & Credential Resolution from Website Vault (Zero Manual Entry)
from src.cloud_sync import get_sync_token
from src.hf_utils import get_hf_token

sync_token = get_sync_token()
os.environ["BIOFLO_SYNC_TOKEN"] = sync_token
os.environ["CEREBRO_SYNC_TOKEN"] = sync_token

hf_tok = get_hf_token()
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_tok
    try:
        from huggingface_hub import login
        login(token=hf_tok, add_to_git_credential=False)
        print("🔑 Hugging Face Hub successfully authenticated dynamically from website vault.")
    except Exception as e_login:
        print(f"ℹ️ Hugging Face login notice: {e_login}")
else:
    print("ℹ️ Hugging Face token not found in website vault.")


## Step 2: Configure & Launch Pretraining
Launches pretraining for `v5_small` (105.9M parameters, 16 shells, 40 shortcut tracts).
- **Hardware Auto-Tuning**: Micro-batches and remat policy are automatically calibrated for optimal TPU MXU throughput.
- **From Scratch / Resume**: Resumes latest checkpoint from `bioflo.in` or local scratch storage with smooth 100-step warmup.
- **Rectified Pretrain Dataset**: 85% FineWeb (sample-10BT) broad web + 15% Code (The Stack v2) with 100% loss-bearing tokens.
- **Remote Sync**: Checkpoints (`latest.pt`, `best.pt`) and live metrics stream directly to `bioflo.in`.

In [ ]:
# ==============================================================================
# ⚡ Step 2: Configure & Launch V5 Small Pretraining
# ==============================================================================
import os, sys, subprocess, importlib

# ── Auto-Sync & Module Cache Purge (Ensures Live Code Updates Without Kernel Restart) ──
REPO_DIR = '/content/cerebro_v5'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith("src") or mod.startswith("training") or mod.startswith("autotune") or mod.startswith("cerebro_autotune"):
        del sys.modules[mod]

from training.train_v5_tpu_jax import run_training

# ── Primary Configuration ─────────────────────────────────────────────────────
MODEL_PRESET       = 'v5_compact'               # 105.9M Parameters (16 Shells, 40 Shortcut Tracts)
FROM_SCRATCH       = False                    # True: Fresh start at Step 0 with random seed 42
AUTO_TUNE          = True                     # Auto-tune discovers micro-batch & remat on TPU v5e-1
EFFECTIVE_BATCH    = 64                       # Global effective batch size (65,536 tok/step at 4096 ctx)
TARGET_TOKENS      = 15_700_000_000          # 15.7 Billion Loss-Bearing Token Target Budget (~60,000 steps)
DATASET_SOURCE     = 'pretrain'                # Rectified 85% FineWeb (sample-10BT) + 15% Code (The Stack v2)
THINKING_CYCLES    = 4                        # Subconscious deliberation cycles per token (increased from 4 to 8)
STAGE_STEPS        = 100_000                  # Steps per context stage (0-100k @ 512, 100k-200k @ 1024, 200k-300k @ 2048, 300k+ @ 4096)
MIN_WINDOW         = 512                      # Minimum initial context window (Stage 0: 512 tokens)

FORCE_TUNE = False

# ── Execute High-Throughput JAX Runner ─────────────────────────────────────────
run_training(
    preset=MODEL_PRESET,
    from_scratch=FROM_SCRATCH,
    auto_tune=AUTO_TUNE,
    force_tune=FORCE_TUNE,                    # Re-run all batch-size candidates instead of using a cached winner
    effective_batch_size=EFFECTIVE_BATCH,
    target_tokens=TARGET_TOKENS,
    dataset_source=DATASET_SOURCE,
    elastic_window=False,
    thinking_cycles=THINKING_CYCLES,
    min_context_window=MIN_WINDOW,
    context_stage_steps=STAGE_STEPS,
    total_window=4096,
    sensory_position_steps=4096,
    max_phase_steps=4096,
    checkpoint_dir="/content/checkpoints/v5_compact_scratch",
    cloud_sync=True,
    cloud_token=get_sync_token(),
    resume_target="latest",
    scmer_lambda=0.05,
    scmer_margin=0.12,
    scmer_subsample=512,
    scmer_pair_samples=8192,
    scmer_interval=8,
    selective_loss=True,
    selective_focal_gamma=0.5,
    selective_noise_cap=8.5,
    z_loss_weight=1e-4,
)


## Step 3: Monitor Checkpoints & Cloud Synchronization
Inspect local checkpoints and live sync status on the bioflo.in remote hub.

In [ ]:
# ==============================================================================
# 📊 Step 3: Checkpoint Inspector & Cloud Hub Dashboard
# ==============================================================================
import json
from src.cloud_sync import check_remote_manifest

print("📁 Local Saved Checkpoints:")
!ls -lh /content/checkpoints/v5_compact_scratch

print("\n🌐 Remote bioflo.in Cloud Hub Status (v5_compact):")
try:
    manifest = check_remote_manifest(model="v5_compact", token=get_sync_token())
    print(json.dumps(manifest, indent=2))
except Exception as e:
    print(f"ℹ️ Cloud sync status notice: {e}")

print("\n📈 Live Web Dashboard: https://bioflo.in/cerebro")


## Step 4: Interactive Live Inference & Prompt Testing
Generate completions from your trained checkpoint (locally cached or auto-pulled from bioflo.in).

In [ ]:
# ==============================================================================
# 💬 Step 4: Interactive Live Inference & Prompt Testing
# ==============================================================================
import os, jax
import jax.numpy as jnp
import numpy as np
from src.jax.v5_model import ModelConfigV5, PRESETS_V5, training_rollout
from src.jax.checkpoint import load_torch_pt_to_jax
from src.cloud_sync import pull_checkpoint
from src.corpus import build_tokenizer, format_stimulus_prompt, clean_motor_response, encode_no_special

CKPT_PATH = "/content/checkpoints/v5_compact_scratch/v5_compact_best.pt"
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = "/content/checkpoints/v5_compact_scratch/v5_compact_latest.pt"
if not os.path.exists(CKPT_PATH):
    from src.jax.checkpoint import discover_latest_checkpoint
    disc_path, _, _ = discover_latest_checkpoint("/content/checkpoints/v5_compact_scratch", preset="v5_compact")
    if disc_path and os.path.exists(disc_path):
        CKPT_PATH = disc_path
    else:
        print("🌐 Checkpoint not found locally. Pulling latest.pt from bioflo.in...")
        pull_checkpoint(CKPT_PATH, target_type="latest", model="v5_compact")

print(f"Loading checkpoint: {CKPT_PATH}...")
tokenizer = build_tokenizer("llama")

cfg_dict = PRESETS_V5["v5_compact"].copy()
cfg_dict["sensory_position_steps"] = 4096
cfg_dict["max_phase_steps"] = 4096
cfg_dict["max_seq_len"] = 4096
config = ModelConfigV5(**cfg_dict)

params, buffers, meta = load_torch_pt_to_jax(CKPT_PATH, config, target_dtype=jnp.bfloat16)
print(f"✓ Loaded model from Step {meta.get('step', 0):,} (Loss: {meta.get('loss', 0.0):.4f})")

def generate(prompt_text: str, max_tokens: int = 128) -> str:
    stimulus = format_stimulus_prompt(prompt_text)
    tokens = encode_no_special(tokenizer, stimulus)
    prompt_arr = jnp.array([tokens], dtype=jnp.int32)
    
    out_logits = training_rollout(
        params=params,
        prompts=prompt_arr,
        target_len=max_tokens,
        buffers=buffers,
        config=config,
        targets=None,
        feedback_tokens=None,
        pure_teacher_forcing=False,
        deliberation_mode="teacher_forcing",
        repetition_penalty=1.15,
    )
    gen_token_ids = np.array(jnp.argmax(out_logits, axis=-1)[0])
    raw = tokenizer.decode(gen_token_ids, skip_special_tokens=False)
    return clean_motor_response(raw).strip()

test_prompt = "What is the capital of France?"
print(f"\nPrompt:   {test_prompt}")
print(f"Response: {generate(test_prompt)}")
